# Working with Excel Files
## Reading, writing, and manipulating Excel files with Python

## 1. Setup and Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 2. Create Sample Excel File

In [ ]:
# Create sample data
sales_data = {
    'Date': pd.date_range('2023-01-01', periods=5),
    'Product': ['A', 'B', 'C', 'A', 'B'],
    'Sales': [100, 150, 120, 110, 160],
    'Quantity': [10, 15, 12, 11, 16]
}

df_sales = pd.DataFrame(sales_data)

# Create another sheet
summary_data = {
    'Product': ['A', 'B', 'C'],
    'Total_Sales': [210, 310, 120],
    'Total_Quantity': [21, 31, 12]
}

df_summary = pd.DataFrame(summary_data)

print("Sales Data:")
print(df_sales)
print("\nSummary Data:")
print(df_summary)

In [ ]:
# Write to Excel with multiple sheets
output_path = '../output/sample_excel.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    df_sales.to_excel(writer, sheet_name='Sales', index=False)
    df_summary.to_excel(writer, sheet_name='Summary', index=False)

print(f"Excel file created at: {output_path}")

## 3. Read Excel Files

In [ ]:
# Read specific sheet
df_read_sales = pd.read_excel(output_path, sheet_name='Sales')
print("Read Sales sheet:")
print(df_read_sales)
print(f"Data types:\n{df_read_sales.dtypes}")

In [ ]:
# Read multiple sheets as dictionary
excel_dict = pd.read_excel(output_path, sheet_name=['Sales', 'Summary'])

print(f"Sheets available: {list(excel_dict.keys())}")
print("\nSales sheet from dict:")
print(excel_dict['Sales'])
print("\nSummary sheet from dict:")
print(excel_dict['Summary'])

In [ ]:
# Read without headers
df_no_header = pd.read_excel(output_path, sheet_name='Sales', header=None)
print("Read without header:")
print(df_no_header)

# Read specific range
df_range = pd.read_excel(output_path, sheet_name='Sales', usecols=[0, 1])
print("\nRead specific columns (0, 1):")
print(df_range)

## 4. Writing with Excel Features

In [ ]:
# Create formatted Excel file
output_formatted = '../output/formatted_excel.xlsx'

with pd.ExcelWriter(output_formatted, engine='openpyxl') as writer:
    # Write data
    df_sales.to_excel(writer, sheet_name='Sales', index=False)
    df_summary.to_excel(writer, sheet_name='Summary', index=False)
    
    # Get workbook and worksheet
    workbook = writer.book
    worksheet_sales = writer.sheets['Sales']
    worksheet_summary = writer.sheets['Summary']
    
    # Auto-adjust column widths
    for column in worksheet_sales.columns:
        max_length = 0
        column_letter = column[0].column_letter
        for cell in column:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        adjusted_width = min(max_length + 2, 50)
        worksheet_sales.column_dimensions[column_letter].width = adjusted_width

print(f"Formatted Excel file created: {output_formatted}")

## 5. Handling Dates and Data Types

In [ ]:
# Read with date parsing
df_dates = pd.read_excel(
    output_path,
    sheet_name='Sales',
    parse_dates=['Date']  # Specify which columns are dates
)

print("Data types with date parsing:")
print(df_dates.dtypes)
print("\nDate column:")
print(df_dates['Date'])

In [ ]:
# Specify data types
df_typed = pd.read_excel(
    output_path,
    sheet_name='Sales',
    dtype={'Product': 'category', 'Sales': 'float64'}
)

print("Data types with dtype specification:")
print(df_typed.dtypes)

## 6. Updating Existing Workbooks

In [ ]:
# Read existing file
df_update = pd.read_excel(output_path, sheet_name='Sales')

# Modify data
df_update['Total'] = df_update['Sales'] * df_update['Quantity']

# Write back to Excel
with pd.ExcelWriter(output_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df_update.to_excel(writer, sheet_name='Sales', index=False)

print("Updated Excel file with new column:")
print(df_update)

## 7. Handling Large Excel Files

In [ ]:
# Read in chunks
chunksize = 2
chunks = []

chunk_iter = pd.read_excel(output_path, sheet_name='Sales', chunksize=chunksize)

for i, chunk in enumerate(chunk_iter):
    print(f"Chunk {i}:")
    print(chunk)
    print()

In [ ]:
# Memory efficient read - read specific columns and rows
df_efficient = pd.read_excel(
    output_path,
    sheet_name='Sales',
    usecols=['Product', 'Sales'],  # Only read specific columns
    nrows=3  # Only read first 3 rows
)

print("Memory efficient read:")
print(df_efficient)

## 8. Excel File Information

In [ ]:
# Get sheet names without reading all data
try:
    xl_file = pd.ExcelFile(output_path)
    print(f"Sheet names: {xl_file.sheet_names}")
    print(f"Number of sheets: {len(xl_file.sheet_names)}")
    
    # Get info about each sheet
    for sheet in xl_file.sheet_names:
        df_temp = pd.read_excel(output_path, sheet_name=sheet)
        print(f"\nSheet '{sheet}':")
        print(f"  Shape: {df_temp.shape}")
        print(f"  Columns: {list(df_temp.columns)}")
except Exception as e:
    print(f"Note: openpyxl may need to be installed for some features. Error: {e}")

## 9. Export to Different Formats

In [ ]:
# Read the data
df = pd.read_excel(output_path, sheet_name='Sales')

# Export to CSV
csv_path = '../output/data.csv'
df.to_csv(csv_path, index=False)
print(f"Data exported to CSV: {csv_path}")

# Export to Parquet (more efficient for large files)
parquet_path = '../output/data.parquet'
df.to_parquet(parquet_path, index=False)
print(f"Data exported to Parquet: {parquet_path}")

# Export to JSON
json_path = '../output/data.json'
df.to_json(json_path, orient='records')
print(f"Data exported to JSON: {json_path}")

# Show file sizes
import os
print(f"\nFile sizes:")
print(f"Excel: {os.path.getsize(output_path) / 1024:.2f} KB")
print(f"CSV: {os.path.getsize(csv_path) / 1024:.2f} KB")
print(f"Parquet: {os.path.getsize(parquet_path) / 1024:.2f} KB")